# Vocabulary & Data Preparation

Convert text → numbers

Add <sos> and <eos> tokens

Prepare tensors

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# Vocabulary Setup
# -----------------------------
vocab = ["<pad>", "<sos>", "<eos>", "i", "like", "ai"]

# Mapping between words and indices
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

def encode(sentence):
    return [word2idx[w] for w in sentence]

def decode(indices):
    return [idx2word[i] for i in indices]

# Example input-output pair
input_sentence = ["i", "like", "ai"]
target_sentence = ["ai", "like", "i"]

# Convert to tensors
input_tensor = torch.tensor([encode(input_sentence)])

# Add <sos> and <eos> for decoder
target_tensor = torch.tensor([
    [word2idx["<sos>"]] + encode(target_sentence) + [word2idx["<eos>"]]
])

# Encoder (LSTM)

Process input sequence

Produce:

All hidden states (important for attention)

Final hidden + cell states

Key difference: we return all outputs, not just last state

In [ ]:
# -----------------------------
# Encoder
# -----------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        # Convert tokens → embeddings
        embedded = self.embedding(x)

        # Pass through LSTM
        outputs, (hidden, cell) = self.lstm(embedded)

        # outputs = all time steps (for attention)
        return outputs, hidden, cell

# Attention Mechanism

Decide which input words to focus on

Compute attention weights

**Key Idea**

At each step:

Compare decoder hidden state with all encoder outputs

Generate importance scores

In [ ]:
# -----------------------------
# Attention
# -----------------------------
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        # Combine encoder + decoder states
        self.energy = nn.Linear(hidden_size * 2, hidden_size)

        # Convert to scalar score
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        seq_len = encoder_outputs.shape[1]

        # Repeat decoder hidden state across input sequence
        hidden = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)

        # Compute energy scores
        energy = torch.tanh(
            self.energy(torch.cat((hidden, encoder_outputs), dim=2))
        )

        # Convert to attention weights
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

# Decoder with Attention

Generate output tokens

Use attention context at every step

**Key Idea**

Decoder input =
[current token embedding + context vector]

In [ ]:
# -----------------------------
# Decoder with Attention
# -----------------------------
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, attention):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Input = embedding + context vector
        self.lstm = nn.LSTM(hidden_size + embed_size, hidden_size, batch_first=True)

        # Output layer
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

        self.attention = attention

    def forward(self, x, hidden, cell, encoder_outputs):
        # Step 1: Embed input token
        embedded = self.embedding(x)

        # Step 2: Compute attention weights
        attn_weights = self.attention(hidden, encoder_outputs)
        attn_weights = attn_weights.unsqueeze(1)

        # Step 3: Compute context vector
        context = torch.bmm(attn_weights, encoder_outputs)

        # Step 4: Combine embedding + context
        lstm_input = torch.cat((embedded, context), dim=2)

        # Step 5: Pass through LSTM
        outputs, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        # Step 6: Predict next token
        predictions = self.fc(torch.cat((outputs, context), dim=2))

        return predictions, hidden, cell

# Seq2Seq Model

Integrate encoder + decoder + attention

Handle teacher forcing

In [ ]:
# -----------------------------
# Seq2Seq
# -----------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        # Encode input
        encoder_outputs, hidden, cell = self.encoder(src)

        outputs = []

        # Start with <sos>
        input_token = trg[:, 0].unsqueeze(1)

        # Decode step-by-step
        for t in range(1, trg.size(1)):
            output, hidden, cell = self.decoder(
                input_token, hidden, cell, encoder_outputs
            )
            outputs.append(output)

            # Teacher forcing
            input_token = trg[:, t].unsqueeze(1)

        return torch.cat(outputs, dim=1)

# Training Setup


Initialize model

Define optimizer and loss

In [ ]:
# -----------------------------
# Training
# -----------------------------
vocab_size = len(vocab)
embed_size = 8
hidden_size = 16

attention = Attention(hidden_size)
encoder = Encoder(vocab_size, embed_size, hidden_size)
decoder = Decoder(vocab_size, embed_size, hidden_size, attention)

model = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop


Learn model parameters

In [ ]:
for epoch in range(300):
    optimizer.zero_grad()

    output = model(input_tensor, target_tensor)

    output = output.view(-1, vocab_size)
    target = target_tensor[:, 1:].reshape(-1)

    loss = criterion(output, target)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.8008
Epoch 50, Loss: 0.0127
Epoch 100, Loss: 0.0029
Epoch 150, Loss: 0.0015
Epoch 200, Loss: 0.0010
Epoch 250, Loss: 0.0007


# Inference (Prediction)

Generate sequence without teacher forcing

In [ ]:
# -----------------------------
# Inference
# -----------------------------
def predict(model, src, max_len=10):
    model.eval()

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)

        input_token = torch.tensor([[word2idx["<sos>"]]])
        result = []

        for _ in range(max_len):
            output, hidden, cell = model.decoder(
                input_token, hidden, cell, encoder_outputs
            )

            pred = output.argmax(2).item()

            if pred == word2idx["<eos>"]:
                break

            result.append(pred)
            input_token = torch.tensor([[pred]])

    return decode(result)

print("Prediction:", predict(model, input_tensor))

Prediction: ['ai', 'like', 'i']
